In [ ]:
import numpy as np
import sys

sys.path.append('../')

from src.data_processor import DataProcessor
from src.model import Word2Vec

Initializing Hyperparameters

In [2]:
NUM_CHARS_TO_READ = 2000000
VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
EPOCHS = 10
BATCH_SIZE = 256
INITIAL_LR = 0.025

Loads Data Preprocessing

In [ ]:
data = DataProcessor('data/text8', vocab_size=VOCAB_SIZE)
data.prepare_data(num_chars=NUM_CHARS_TO_READ)

Model Initialization

In [4]:
model = Word2Vec(vocab_size=data.vocab_size, embedding_dim=EMBEDDING_DIM)

Training Utilities

In [5]:
total_words = len(data.corpus_ids)
total_batches = (total_words * data.window_size * 2)

Evaluation Metric

In [6]:
def get_similar_words(model, data, word, k=5):
    """Calculates cosine similarity to find the closest words"""
    if word not in data.word2idx:
        return f"'{word}' is out of vocabulary."
    
    word_idx = data.word2idx[word]
    
    combined_embeddings = model.W_in + model.W_out
    target_vec = combined_embeddings[word_idx]
    
    dot_products = np.dot(combined_embeddings, target_vec)
    norms = np.linalg.norm(combined_embeddings, axis=1) * np.linalg.norm(target_vec)
    similarities = dot_products / norms
    
    closest_idxs = np.argsort(similarities)[-(k+1):][::-1]
    
    results = []
    for idx in closest_idxs:
        if idx != word_idx and idx != 0: 
            results.append((data.idx2word[idx], similarities[idx]))
            if len(results) == k:
                break
    return results

### Training Loop <br>
**Iterates through mini-batches using a memory-efficient generator. Applies SGD with learning rate decay and evaluates semantic progression per epoch**

In [7]:
test_words = ['one', 'first', 'state', 'he']

for epoch in range(EPOCHS):
    epoch_loss = 0
    batch_count = 0
    
    for centers, contexts, negatives in data.generate_batches(BATCH_SIZE):
        progress = (epoch * total_batches + batch_count) / (EPOCHS * total_batches)
        current_lr = max(INITIAL_LR * (1 - progress), INITIAL_LR * 0.0001)
        
        loss = model.forward_backward(centers, contexts, negatives, learning_rate=current_lr)
        epoch_loss += loss
        batch_count += 1
        
        if batch_count % 1000 == 0:
            print(f"Epoch: {epoch+1}/{EPOCHS} | Batch: {batch_count} | Loss: {loss:.4f} | LR: {current_lr:.4f}")
    
    print("Evaluation:")
    for word in test_words:
        sims = get_similar_words(model, data, word, k=3)
        formatted_sims = ", ".join([f"{w} ({s:.2f})" for w, s in sims])
        print(f"Closest to '{word}': {formatted_sims}")
    print("-" * 40)

Epoch: 1/10 | Batch: 1000 | Loss: 2.8288 | LR: 0.0250
Evaluation:
    Closest to 'one': the (0.96), a (0.96), zero (0.95)
    Closest to 'first': the (0.85), one (0.85), s (0.85)
    Closest to 'state': not (0.73), six (0.70), had (0.68)
    Closest to 'he': of (0.89), zero (0.88), a (0.88)
----------------------------------------
Epoch: 2/10 | Batch: 1000 | Loss: 2.4887 | LR: 0.0225
Evaluation:
    Closest to 'one': zero (0.91), nine (0.91), a (0.90)
    Closest to 'first': were (0.84), s (0.84), as (0.83)
    Closest to 'state': being (0.75), six (0.75), has (0.74)
    Closest to 'he': six (0.86), or (0.85), zero (0.85)
----------------------------------------
Epoch: 3/10 | Batch: 1000 | Loss: 2.2977 | LR: 0.0200
Evaluation:
    Closest to 'one': zero (0.86), nine (0.85), three (0.84)
    Closest to 'first': were (0.80), from (0.78), six (0.77)
    Closest to 'state': being (0.71), has (0.70), form (0.69)
    Closest to 'he': six (0.78), nine (0.77), government (0.77)
---------------

We see the Loss Convergence strarting from 2.8 and settling aroung 2.07 which is a great sign. Also, from epoch 5 we see a clear semantic context, "first" associates with "second", and "state" with "alaska" and "february". "He" clusters with "argued", matching grammatical patterns for subjects and verbs. We need to keep in mind, that the data set is relatively small

---

Test out-of-sample words to inspect the final vector space

In [8]:
word_to_test = "computer"
print(f"Words closest to '{word_to_test}':")
get_similar_words(model, data, word_to_test, k=3)

Words closest to 'computer':


[('solved', np.float64(0.5472773385763943)),
 ('encountered', np.float64(0.49490035769064694)),
 ('champions', np.float64(0.4931906091076669))]